In [1]:
!python download_ckpts.py

ModernBERT_T2000.safetensors: 100%|███████| 1.61G/1.61G [00:17<00:00, 92.6MB/s]
Direct_ModernBERT_T2000.safetensors: 100%|█| 1.61G/1.61G [00:18<00:00, 88.9MB/s


In [2]:
from model import Baseline
from train_pdnc import *

def to_device(data: dict, device: torch.device) -> dict:
    """Move all tensors in a dict (possibly nested / in lists) to `device`."""
    out = {}
    for k, v in data.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device)
        elif isinstance(v, (list, tuple)):
            out[k] = type(v)(
                item.to(device) if isinstance(item, torch.Tensor) else item
                for item in v
            )
        elif isinstance(v, dict):
            out[k] = to_device(v, device)
        else:
            out[k] = v
    return out

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import pandas as pd
NOVELS = ['PrideAndPrejudice',
 'AgeOfInnocence',
 'HandfulOfDust',
 'TheMysteriousAffairAtStyles',
 'SenseAndSensibility',
 'Emma',
 'NorthangerAbbey',
 'TheSportOfTheGods',
 'SignOfFour',
 'NightAndDay',
 'ThePictureOfDorianGray',
 'AnneOfGreenGables',
 'TheInvisibleMan',
 'TheGambler',
 'TheSunAlsoRises',
 'TheManWhoWasThursday',
 'DaisyMiller',
 'HowardsEnd',
 'Persuasion',
 'AliceInWonderland',
 'ARoomWithAView',
 'TheAwakening']


DATA_PATH = '../data/pdnc_source/'

dfs = {}
for f in NOVELS: 
    dfs[f] = pd.read_csv(f'{DATA_PATH}/{f}/quote_info.csv')

qid2type = {}

for novel, df in dfs.items() : 
    qids = [novel + '_' + i for i in df['qID'].values]
    qTypes = df['qType'].values
    for k,v in zip(qids, qTypes) : 
        qid2type[k] = v

In [7]:
config = yaml.safe_load(open('pdnc_config.yaml'))

In [8]:
ckpt = 'ckpts/ModernBERT_T2000.safetensors'
ckpt_indiv = 'ckpts/Direct_ModernBERT_T2000.safetensors'


config['model']['anaphora_pred'] = False
model = Baseline(config['model'])
model_indiv = Baseline(config['model'])

control = Baseline(config['model'])


In [9]:
from safetensors.torch import load_file

ckpt = load_file(ckpt)
ckpt_indiv = load_file(ckpt_indiv)

In [10]:
model.load_state_dict(ckpt, strict=True)
model_indiv.load_state_dict(ckpt_indiv, strict=True)


<All keys matched successfully>

In [15]:
NAME_KEY = "Components_N2000_S512_K200_ModernBERT_Large.pkl"

train_dataset = PDNCDataset(
            graph_path = '../data/pdnc_source',
            split='test',
            split_num=3,
            filter=True,
        name_key=NAME_KEY
        )

Using data at ../data/pdnc_source/HandfulOfDust/Components_N2000_S512_K200_ModernBERT_Large.pkl
Using data at ../data/pdnc_source/PrideAndPrejudice/Components_N2000_S512_K200_ModernBERT_Large.pkl
Using data at ../data/pdnc_source/TheInvisibleMan/Components_N2000_S512_K200_ModernBERT_Large.pkl
Using data at ../data/pdnc_source/TheSportOfTheGods/Components_N2000_S512_K200_ModernBERT_Large.pkl


In [16]:
len(train_dataset)

231

In [17]:
collate_fn = partial(collate_fn, data_args={'pad_token_id':50283})

In [18]:
loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn = collate_fn)
# indiv_loader = torch.utils.data.DataLoader(indiv_dataset, batch_size=16, collate_fn = collate_fn)

In [19]:
device = torch.device('cuda:1')

model.eval()
model = model.to(device)

model_indiv.eval()
model_indiv = model_indiv.to(device)

control.eval()
control = control.to(device)


In [20]:
import quote_speaker_analysis as qsa

In [21]:
def get_acc(windows) : 
    for w in windows : 
        preds = [w.mention_speaker_ids[i] for i in w.predicted_m]
        acc = [i==j for i,j in zip(preds, w.quote_speaker_ids)]
        w.acc = acc
    

windows = []
windows_i = []
windows_control = []
with torch.no_grad() : 

    for idx, batch in enumerate(tqdm((loader))): 
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16) :
            batch = to_device(batch, device)
            labels = batch.pop('candidate_labels')
            out = qsa.generic_quote_mention_reps(model, batch['g'], batch['input_ids'], batch['att_mask'], batch['st'], batch['et'], None, score=True)
            out_indiv = qsa.generic_quote_mention_reps(model_indiv, batch['g'], batch['input_ids'], batch['att_mask'], batch['st'], batch['et'], None, score=True)
            out_c= qsa.generic_quote_mention_reps(control, batch['g'], batch['input_ids'], batch['att_mask'], batch['st'], batch['et'], None)

            for bs, w in enumerate(out_indiv) : 
                qids = [w.quote_ids[i] for i in batch['g'].is_pred[bs]]
                w.quote_types = [qid2type[q] for q in qids]
                
            for bs, w in enumerate(out) : 
                qids = [w.quote_ids[i] for i in batch['g'].is_pred[bs]]
                w.quote_types = [qid2type[q] for q in qids]
                
            for bs, w in enumerate(out_c) : 
                qids = [w.quote_ids[i] for i in batch['g'].is_pred[bs]]
                w.quote_types = [qid2type[q] for q in qids]

            get_acc(out)
            get_acc(out_indiv)
                        
            windows.extend(out)
            windows_i.extend(out_indiv)
            windows_control.extend(out_c)

            # if idx == 2: 
            #     break
        # out_indiv = generic_quote_mention_reps(model_indiv, batch['g'], batch['input_ids'], batch['att_mask'], batch['st'], batch['et'], get_spk_ms)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:45<00:00,  3.01s/it]


In [23]:
len(windows), len(windows_i), len(windows_control)

(231, 231, 231)

### Separability Analysis

In [24]:
x = list(zip(list(range(0, 1801, 200)), range(200, 2001, 200)))
quote_types = ['Explicit', 'Implicit', 'Anaphoric']

In [25]:
# Q-Q
binned_sweep = qsa.sweep_cohens_d_over_distance_bins_by_quote_type( windows, windows_i, bins=x,comparison_fn=qsa.run_rsa_comparison,quote_types=quote_types, control_windows= windows_control)

In [26]:
# Q-M
binned_sweep_m = qsa.sweep_cohens_d_over_distance_bins_by_quote_type(windows, windows_i, bins=x,comparison_fn=qsa.run_rsa_mention_comparison,quote_types=quote_types, control_windows= windows_control)

In [27]:
# M-M Anaphora
bins = list(zip(list(range(0, 281, 20)), range(20, 301, 20)))
k = 2
binned_sweep_mn = qsa.run_nearest_mention_mention_rsa_by_bin(windows, windows_i, bins, negative_k=k, control_windows= windows_control, direction='backward')


#### plots

In [29]:
qsa.plot_cohens_d_by_bin_per_quote_type_with_overall(binned_sweep_m, binned_sweep_mn, title='', out_path='plot_1.pdf' )

In [30]:
qsa.plot_cohens_d_by_bin_per_quote_type_with_mean_similarity(binned_sweep, binned_sweep_mn, title='', out_path='plot_2.pdf' )

### Correlation Between Downstream Accuracy and Q-M Separability

In [35]:
bins = [(i, i + 200) for i in range(0, 2000, 200)]
quote_types = ["Explicit", "Anaphoric", "Implicit"]

points = qsa.collect_accuracy_and_qm_separability_by_bin(
    windows, windows_i, quote_types, bins,
    control_windows=None,  # optional, e.g. ModernBERT
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 215/215 [00:12<00:00, 17.43it/s]


In [36]:
qsa.plot_accuracy_vs_separability_by_quote_type(points,  "per_qt_regs.pdf", )